# Pipeline V8 — Fase 3 Legado: Re-rotulação Databricks (p=20, 11.542 estados)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-007). **Execução: Databricks.**

Re-calcula `melhor_jogada` e `score_melhor_jogada` somente para estados onde o
rótulo Minimax p=11 é potencialmente subótimo:

```
critério: arestas_livres > 11  E  prof_min > 11
onde:     arestas_livres = 31 - qtd_tracos
          prof_min       = total_caixas_cadeias_longas + 2 × qtd_cadeias_longas
```

**Profundidade única**: `p=20`. O Minimax para naturalmente quando o jogo termina;
p=20 é suficiente para qualquer estado com ≤ 19 arestas livres (máximo observado).

**Estados a re-rotular** (análise 100% do dataset — 2026-05-14):

| arestas_livres | Estados |
|---|---|
| 12 | 3.601 |
| 13 | 3.000 |
| 14 | 2.232 |
| 15 | 1.581 |
| 16 | 836 |
| 17 | 244 |
| 18 | 45 |
| 19 | 3 |
| **Total** | **11.542** (1,52%) |

**Pré-requisito**: Fase 2 V8 concluída (NPZs com schema v2-a3 — campos
`qtd_cadeias_longas` e `total_caixas_cadeias_longas` presentes).

**Auto-contido**: sem imports de módulos externos. Todo o Bitboard está inline
na função `processar_lote_pandas`, idêntico ao padrão do HighPerf.

**Sobrescrita atômica**: `.tmp.npz` + `os.replace()`. Todos os 14 campos v2-a3
são preservados no write-back.

**Após a execução**: verificar a auditoria e deletar este notebook.

> **⚠️ ALERTA DE MANUTENÇÃO: BUGS ALGÓRITMICOS NO BITBOARD (Maio/2026)**
>
> Este algoritmo Minimax via Bitboard possui duas particularidades matemáticas gravíssimas que foram corrigidas na versão atual, mas que **não devem ser alteradas ou esquecidas** caso esse código seja adaptado para outros formatos:
> 
> 1. **Caixas Pré-Fechadas (Falsos Positivos):** Como a matriz vira bits, adicionar um traço não pode contar simplesmente se a máscara da caixa ficou completa `(edges | bit) & mask == mask`. É OBRIGATÓRIO checar se a caixa *não estava fechada antes* `and (edges & mask) != mask`. Sem isso, o Bitboard injeta pontos fantasmas durante a simulação em profundidade e corrompe o score.
> 2. **Offsets na Poda Alpha-Beta Incremental:** Diferente de motores comuns, este Bitboard retorna *scores incrementais* em vez de absolutos. Por causa disso, ao chamar a subárvore no `solve_minimax_bitboard`, nós **temos que repassar os limites `alpha` e `beta` com um offset de `- closed` (ou `+ closed` pro adversário)**. Se você repassar o `alpha` puro, a subárvore sofre "poda prematura" e o motor escolhe jogadas perdedoras por achar que metas inalcançáveis não valem a pena serem processadas.

In [ ]:
# Configuração — ajustar paths conforme montagem no Databricks.

# Path do diretório NPZ no Workspace do Databricks.
DIR_NPZ = '/Workspace/Users/diondu@gmail.com/CNN/profundidade_minimax_11_v7_adaptativo'

# Profundidade única de re-rotulação.
DEPTH_REROTULACAO = 20

# Lote de NPZs por job Spark (checkpoint granularity).
LOTE_ARQUIVOS = 4

# Se False: só re-rotula estados onde depth_melhor_jogada < DEPTH_REROTULACAO.
# Se True:  re-rotula todos os estados elegíveis, mesmo os já re-rotulados.
FORCAR_REGRAVAR = False

print(f'DIR_NPZ            = {DIR_NPZ}')
print(f'DEPTH_REROTULACAO  = {DEPTH_REROTULACAO}')
print(f'LOTE_ARQUIVOS      = {LOTE_ARQUIVOS}')
print(f'FORCAR_REGRAVAR    = {FORCAR_REGRAVAR}')

In [ ]:
# Imports — apenas bibliotecas padrão/Databricks. Nenhum módulo externo.

import os
import glob
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# `spark` está disponível automaticamente no cluster Databricks.

SUFIXOS_SIMETRIA = ('_refH', '_refV', '_r180')

def _e_original(path: str) -> bool:
    stem = os.path.splitext(os.path.basename(path))[0]
    return not any(stem.endswith(s) for s in SUFIXOS_SIMETRIA)

arquivos_originais = sorted([
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if _e_original(f)
])

print(f'NPZs originais encontrados: {len(arquivos_originais)}')
print('Imports OK.')

In [ ]:
# Diagnóstico: contar estados elegíveis por arestas_livres.
# Gate T-A3-007: confirmar distribuição antes de rodar.
# Esperado: 12→3.601, 13→3.000, 14→2.232, 15→1.581, 16→836, 17→244, 18→45, 19→3 (total=11.542)

contagem_por_arestas = Counter()
n_total_elegíveis = 0

for arq in arquivos_originais:
    with np.load(arq, allow_pickle=False) as d:
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    elegível = (arestas_livres > 11) & (prof_min > 11)

    for al in arestas_livres[elegível]:
        contagem_por_arestas[int(al)] += 1
    n_total_elegíveis += int(elegível.sum())

n_total_estados = len(arquivos_originais) * 5000  # estimativa
print(f'Estados elegíveis  : {n_total_elegíveis:,}')
print(f'Percentual         : {100 * n_total_elegíveis / n_total_estados:.2f}%')
print()
print(f'{"arestas_livres":>16} | {"estados":>8}')
print('-' * 30)
for al in sorted(contagem_por_arestas):
    print(f'{al:>16} | {contagem_por_arestas[al]:>8}')
print(f'{"TOTAL":>16} | {n_total_elegíveis:>8}')

In [ ]:
# Worker Spark: Bitboard inline, profundidade DEPTH_REROTULACAO.
# Recebe iterador de DataFrames com coluna `estado_bytes BINARY`.
# Retorna DataFrame com `estado_bytes, melhor_jogada, scores`.
#
# Bugs corrigidos (ver alerta acima):
#   Bug 1: `edges & bm != bm` — não conta caixas pré-fechadas.
#   Bug 2: offset alpha/beta — `alpha - closed` / `beta + closed` quando o
#          mesmo jogador captura caixas e continua jogando.

DEPTH_WORKER = DEPTH_REROTULACAO  # capturado no closure do driver

def processar_lote_pandas(iterator):
    import pandas as pd
    import numpy as np
    import random

    # 1. Reconstrói tabelas Bitboard localmente no worker (evita pickle)
    h, w = 9, 7
    edge_to_bit = {}
    bit_to_label = {}
    bit_idx = 0
    for r in range(h):
        for c in range(w):
            if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                edge_to_bit[(r, c)] = bit_idx
                bit_to_label[bit_idx] = f'H_{r}_{c}' if r % 2 == 0 else f'V_{r}_{c}'
                bit_idx += 1

    n_edges = bit_idx   # deve ser 31
    all_mask = (1 << n_edges) - 1

    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            mask = ((1 << edge_to_bit[(r-1, c)]) | (1 << edge_to_bit[(r+1, c)]) |
                    (1 << edge_to_bit[(r, c-1)]) | (1 << edge_to_bit[(r, c+1)]))
            box_masks.append(mask)

    # edge_boxes[b] = tupla das máscaras de caixa que contêm a aresta b
    edge_boxes = [tuple(bm for bm in box_masks if bm & (1 << b)) for b in range(n_edges)]

    # 2. Minimax Bitboard com TT completa e ambos os bugs corrigidos
    def solve_minimax_bitboard(edges, depth, alpha, beta, maximizing, tt):
        if depth == 0 or edges == all_mask:
            return 0

        tt_key = (edges, depth, maximizing)
        if tt_key in tt:
            flag, val = tt[tt_key]
            if flag == 0: return val
            if flag == 1 and val >= beta: return val
            if flag == 2 and val <= alpha: return val

        moves = []
        for i in range(n_edges):
            if not (edges & (1 << i)):
                ne = edges | (1 << i)
                # Bug 1 fix: só conta caixa se NÃO estava fechada antes
                closed = sum(1 for bm in edge_boxes[i] if ne & bm == bm and edges & bm != bm)
                moves.append((i, closed))
        moves.sort(key=lambda x: x[1], reverse=True)  # move ordering

        orig_alpha = alpha
        orig_beta = beta
        best_val = -10000 if maximizing else 10000

        for bit, closed in moves:
            new_e = edges | (1 << bit)
            if maximizing:
                if closed > 0:
                    # Bug 2 fix: offset alpha/beta pelo número de caixas capturadas
                    val = closed + solve_minimax_bitboard(
                        new_e, depth - 1, alpha - closed, beta - closed, True, tt
                    )
                else:
                    val = solve_minimax_bitboard(new_e, depth - 1, alpha, beta, False, tt)
                best_val = max(best_val, val)
                alpha = max(alpha, best_val)
            else:
                if closed > 0:
                    val = -closed + solve_minimax_bitboard(
                        new_e, depth - 1, alpha + closed, beta + closed, False, tt
                    )
                else:
                    val = solve_minimax_bitboard(new_e, depth - 1, alpha, beta, True, tt)
                best_val = min(best_val, val)
                beta = min(beta, best_val)
            if beta <= alpha:
                break

        # Gravação na TT com flag correto
        if maximizing:
            flag = 2 if best_val <= orig_alpha else (1 if best_val >= beta else 0)
        else:
            flag = 1 if best_val >= orig_beta else (2 if best_val <= alpha else 0)
        tt[tt_key] = (flag, best_val)
        return best_val

    # 3. Processamento de cada DataFrame recebido do Spark
    for pdf in iterator:
        resultados = []
        for estado_bytes in pdf['estado_bytes']:
            # Converte matriz (9,7) {0,1,8,9} → bitmask de 31 bits
            mat = np.frombuffer(estado_bytes, dtype=np.int8).reshape(9, 7)
            edges = 0
            idx = 0
            for r in range(9):
                for c in range(7):
                    if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                        if mat[r, c] == 9:
                            edges |= (1 << idx)
                        idx += 1

            # TT compartilhada por tabuleiro (entre as 31 jogadas raiz)
            tt = {}
            scores = np.full(31, -1_000_000_000.0, dtype=np.float32)

            for i in range(n_edges):
                if not (edges & (1 << i)):
                    new_e = edges | (1 << i)
                    closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm and edges & bm != bm)
                    if closed > 0:
                        res = closed + solve_minimax_bitboard(
                            new_e, DEPTH_WORKER - 1, -10001, 10001, True, tt
                        )
                    else:
                        res = solve_minimax_bitboard(
                            new_e, DEPTH_WORKER - 1, -10001, 10001, False, tt
                        )
                    scores[i] = float(res)

            # Argmax (desempate aleatório entre ótimas empatadas)
            validos = [i for i, s in enumerate(scores) if s > -1e8]
            if not validos:
                melhor_rotulo = ''
            else:
                max_s = float(np.max(scores[list(validos)]))
                best_idx = random.choice([i for i in validos if scores[i] == max_s])
                melhor_rotulo = bit_to_label[best_idx]

            resultados.append({
                'estado_bytes': estado_bytes,
                'melhor_jogada': melhor_rotulo,
                'scores': scores.tolist(),
            })
        yield pd.DataFrame(resultados)


print(f'Worker pronto. DEPTH_WORKER = {DEPTH_WORKER}')

In [ ]:
# Loop principal: build queue → Spark → write-back atômico.
#
# Critério de pendência: elegível E (FORCAR_REGRAVAR OU depth_melhor_jogada < DEPTH_REROTULACAO)
# Write-back: preserva TODOS os 14 campos v2-a3.

spark.conf.set('spark.databricks.execution.timeout', 7200)
schema_spark = 'estado_bytes BINARY, melhor_jogada STRING, scores ARRAY<FLOAT>'

# Identificar arquivos que têm ao menos 1 estado elegível e pendente
arquivos_pendentes = []
for arq in arquivos_originais:
    with np.load(arq, allow_pickle=False) as d:
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_mj   = d['depth_melhor_jogada'].astype(np.int32)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    elegível = (arestas_livres > 11) & (prof_min > 11)
    pendente = elegível & (FORCAR_REGRAVAR or (depth_mj < DEPTH_REROTULACAO))

    if pendente.any():
        arquivos_pendentes.append(arq)

print(f'Arquivos com estados pendentes: {len(arquivos_pendentes)}')

# Processar em lotes
for i in range(0, len(arquivos_pendentes), LOTE_ARQUIVOS):
    lote = arquivos_pendentes[i : i + LOTE_ARQUIVOS]
    estados_unicos = set()

    # Coletar estados pendentes únicos do lote
    for arq in lote:
        with np.load(arq, allow_pickle=False) as d:
            estados    = d['estados']
            qtd_tracos = d['qtd_tracos'].astype(np.int32)
            qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
            total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)
            depth_mj   = d['depth_melhor_jogada'].astype(np.int32)

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        elegível = (arestas_livres > 11) & (prof_min > 11)
        pendente = elegível & (FORCAR_REGRAVAR or (depth_mj < DEPTH_REROTULACAO))

        for j in np.where(pendente)[0]:
            estados_unicos.add(estados[j].tobytes())

    lote_num = i // LOTE_ARQUIVOS + 1
    total_lotes = (len(arquivos_pendentes) + LOTE_ARQUIVOS - 1) // LOTE_ARQUIVOS
    print(f'--- Lote {lote_num}/{total_lotes}: {len(lote)} arquivos, {len(estados_unicos)} estados únicos ---')

    t0 = time.time()

    # Execução distribuída
    pdf_in = pd.DataFrame({'estado_bytes': list(estados_unicos)})
    df_spark = spark.createDataFrame(pdf_in)

    try:
        n_particoes = int(spark.conf.get('spark.sql.shuffle.partitions', '200'))
    except (ValueError, Exception):
        n_particoes = 200
    df_spark = df_spark.repartition(n_particoes)

    df_result = df_spark.mapInPandas(processar_lote_pandas, schema=schema_spark)
    resultados_lote = df_result.collect()

    cache = {
        bytes(row.estado_bytes): (row.melhor_jogada, np.array(row.scores, dtype=np.float32))
        for row in resultados_lote
    }
    print(f'  Spark: {time.time() - t0:.1f}s. Gravando...')

    # Write-back atômico — preserva TODOS os campos v2-a3
    for arq in lote:
        with np.load(arq, allow_pickle=False) as f:
            d = dict(f)

        estados    = d['estados']
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_mj   = d['depth_melhor_jogada'].astype(np.int32)

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        elegível = (arestas_livres > 11) & (prof_min > 11)
        pendente = elegível & (FORCAR_REGRAVAR or (depth_mj < DEPTH_REROTULACAO))
        indices = np.where(pendente)[0]

        mj_arr  = d['melhor_jogada'].copy()
        smj_arr = d['score_melhor_jogada'].copy()
        dmj_arr = d['depth_melhor_jogada'].copy()

        for j in indices:
            chave = estados[j].tobytes()
            rotulo, scores = cache[chave]
            mj_arr[j]    = rotulo
            smj_arr[j]   = scores
            dmj_arr[j]   = np.int8(DEPTH_REROTULACAO)

        # Preservar todos os campos v2-a3 no NPZ de saída
        d['melhor_jogada']       = mj_arr
        d['score_melhor_jogada'] = smj_arr
        d['depth_melhor_jogada'] = dmj_arr

        tmp = arq + '.tmp.npz'
        np.savez_compressed(tmp, **d)
        os.replace(tmp, arq)

        print(f'  {os.path.basename(arq)}: {len(indices)} re-rotulados.')

print('\nTodo o trabalho foi finalizado com sucesso!')

In [ ]:
# Auditoria pós-re-rotulação.
# Verifica: todos os estados elegíveis têm depth_melhor_jogada == DEPTH_REROTULACAO.
# Nenhum .tmp residual.

erros = []
tmps_residuais = []
n_rerotulados_total = 0

for arq in arquivos_originais:
    tmp = arq + '.tmp.npz'
    if os.path.exists(tmp):
        tmps_residuais.append(os.path.basename(tmp))

    with np.load(arq, allow_pickle=False) as d:
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_mj   = d['depth_melhor_jogada'].astype(np.int32)
        mj         = d['melhor_jogada']

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    elegível = (arestas_livres > 11) & (prof_min > 11)

    n_rerotulados_total += int(elegível.sum())

    # Todo estado elegível deve ter depth >= DEPTH_REROTULACAO e melhor_jogada preenchida
    for j in np.where(elegível)[0]:
        if depth_mj[j] < DEPTH_REROTULACAO:
            erros.append(f'{os.path.basename(arq)}[{j}]: depth={depth_mj[j]} < {DEPTH_REROTULACAO}')
        if not str(mj[j]):
            erros.append(f'{os.path.basename(arq)}[{j}]: melhor_jogada vazia')

print(f'Estados elegíveis auditados: {n_rerotulados_total:,}')

if tmps_residuais:
    print(f'FALHA: {len(tmps_residuais)} arquivo(s) .tmp residual(is):')
    for t in tmps_residuais[:5]:
        print(f'  {t}')

if erros:
    print(f'FALHA: {len(erros)} estado(s) com problema:')
    for e in erros[:10]:
        print(f'  {e}')

if not tmps_residuais and not erros:
    print(f'OK: {n_rerotulados_total:,} estados elegíveis com depth>={DEPTH_REROTULACAO} e melhor_jogada preenchida.')
    print('Nenhum .tmp residual.')
    print('Pronto para deletar este notebook e executar fase4_augmentacao_simetria.ipynb.')